# Supply Chain Data Analytics — AICTE Internship
**Dataset:** Kaggle E-Commerce Shipping Data (`Train.csv`)  
**Objective:** Load, clean, explore, and visualise the dataset to uncover actionable delivery insights.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# BLOCK 1 — Imports
# ─────────────────────────────────────────────────────────────────────────────
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')           # suppress non-critical warnings
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 110            # crisp inline figures

print('Libraries loaded successfully.')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# BLOCK 2 — Load Dataset
# ─────────────────────────────────────────────────────────────────────────────
CSV_PATH = 'Train.csv'          # adjust path if needed

df = pd.read_csv(CSV_PATH)

print(f'Shape  : {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'Columns: {df.columns.tolist()}')
df.head()

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# BLOCK 3 — Data Cleaning
# ─────────────────────────────────────────────────────────────────────────────

# 3-A  Drop the 'ID' column — it is a row identifier with zero predictive power
df.drop(columns=['ID'], inplace=True)
print('▶ Dropped column: ID')

# 3-B  Inspect missing values across every remaining column
missing = df.isnull().sum()
print('\n▶ Missing values per column:')
print(missing[missing >= 0].to_string())     # show all, zeros included for clarity

# 3-C  Handle missing values (if any are discovered)
# Strategy:
#   • Numerical columns  → fill with the column MEDIAN (robust to outliers)
#   • Categorical columns → fill with the column MODE
num_cols = df.select_dtypes(include='number').columns.tolist()
cat_cols = df.select_dtypes(include='object').columns.tolist()

for col in num_cols:
    if df[col].isnull().any():
        df[col].fillna(df[col].median(), inplace=True)
        print(f'  Filled numerical NaNs in "{col}" with median ({df[col].median():.2f})')

for col in cat_cols:
    if df[col].isnull().any():
        df[col].fillna(df[col].mode()[0], inplace=True)
        print(f'  Filled categorical NaNs in "{col}" with mode ("{df[col].mode()[0]}")')

# 3-D  Final sanity check
remaining_nulls = df.isnull().sum().sum()
print(f'\n▶ Total nulls remaining after cleaning: {remaining_nulls}')
print(f'▶ Clean dataset shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
df.dtypes

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# BLOCK 4 — EDA: Correlation Matrix Heatmap
# ─────────────────────────────────────────────────────────────────────────────
# Select only numerical columns so the correlation makes statistical sense.
# 'Reached.on.Time_Y.N' (0 = on-time, 1 = NOT on time) is the target variable.

corr_matrix = df[num_cols].corr()

fig, ax = plt.subplots(figsize=(10, 7))

sns.heatmap(
    corr_matrix,
    annot=True,           # print the coefficient inside each cell
    fmt='.2f',            # 2 decimal places
    cmap='coolwarm',      # blue = negative, red = positive correlation
    center=0,             # centre the colour scale at 0
    linewidths=0.5,
    ax=ax
)

ax.set_title('Correlation Matrix — Numerical Features\n(Target: Reached.on.Time_Y.N)',
             fontsize=13, pad=14)
plt.tight_layout()
plt.savefig('heatmap_correlation.png', bbox_inches='tight')
plt.show()

# Programmatically surface the top correlators with the target
target_corr = (
    corr_matrix['Reached.on.Time_Y.N']
    .drop('Reached.on.Time_Y.N')        # exclude self-correlation
    .abs()
    .sort_values(ascending=False)
)
print('\n▶ Feature correlation strength with target (|r|):')
print(target_corr.to_string())

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# BLOCK 5 — Visualisation 1: Countplot
#           Customer Care Calls vs On-Time Delivery
# ─────────────────────────────────────────────────────────────────────────────
# Rationale: If customers calling more times correlates with late deliveries,
# it signals operational bottlenecks worth investigating.

fig, ax = plt.subplots(figsize=(10, 5))

sns.countplot(
    data=df,
    x='Customer_care_calls',
    hue='Reached.on.Time_Y.N',
    palette={0: '#2ecc71', 1: '#e74c3c'},   # green = on-time, red = late
    edgecolor='white',
    ax=ax
)

# Annotate each bar with its count for quick reading
for patch in ax.patches:
    height = patch.get_height()
    if height > 0:
        ax.text(
            patch.get_x() + patch.get_width() / 2,
            height + 15,
            f'{int(height):,}',
            ha='center', va='bottom', fontsize=8
        )

ax.set_title('Customer Care Calls vs On-Time Delivery', fontsize=13)
ax.set_xlabel('Number of Customer Care Calls')
ax.set_ylabel('Number of Orders')

# Rename legend labels for clarity
handles, _ = ax.get_legend_handles_labels()
ax.legend(handles, ['On Time (0)', 'Late (1)'], title='Delivery Status')

plt.tight_layout()
plt.savefig('countplot_care_calls.png', bbox_inches='tight')
plt.show()

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# BLOCK 6 — Visualisation 2: Boxplot
#           Discount Offered vs On-Time Delivery
# ─────────────────────────────────────────────────────────────────────────────
# Rationale: Large discounts may be applied to high-demand items that are
# harder to fulfil on time — the spread of discount percentages can reveal this.

fig, ax = plt.subplots(figsize=(8, 5))

# Cast the target to string so the palette dict keys ('0'/'1') are matched
# correctly by seaborn ≥ 0.12 when the column holds integer values.
df_plot = df.assign(**{'Reached.on.Time_Y.N': df['Reached.on.Time_Y.N'].astype(str)})

sns.boxplot(
    data=df_plot,
    x='Reached.on.Time_Y.N',
    y='Discount_offered',
    palette={'0': '#2ecc71', '1': '#e74c3c'},   # string keys for seaborn ≥ 0.12
    order=['0', '1'],
    width=0.45,
    flierprops=dict(marker='o', markerfacecolor='grey', markersize=3, alpha=0.4),
    ax=ax
)

ax.set_title('Discount Offered vs On-Time Delivery', fontsize=13)
ax.set_xlabel('Delivery Status  (0 = On Time, 1 = Late)')
ax.set_ylabel('Discount Offered (%)')

# Overlay median labels for precision
medians = df.groupby('Reached.on.Time_Y.N')['Discount_offered'].median()
for tick, (status, median_val) in enumerate(medians.items()):
    ax.text(tick, median_val + 0.5, f'Median: {median_val:.1f}%',
            ha='center', va='bottom', fontsize=9, color='black', fontweight='bold')

plt.tight_layout()
plt.savefig('boxplot_discount.png', bbox_inches='tight')
plt.show()

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# BLOCK 7 — Visualisation 3: Pie Chart
#           Distribution of Orders Across Warehouse Blocks
# ─────────────────────────────────────────────────────────────────────────────
# Rationale: Uneven warehouse load distribution can be a root cause of
# late deliveries — overloaded blocks struggle with order fulfilment speed.

warehouse_counts = df['Warehouse_block'].value_counts()

fig, ax = plt.subplots(figsize=(7, 7))

wedge_props = dict(edgecolor='white', linewidth=1.5)

wedges, texts, autotexts = ax.pie(
    warehouse_counts,
    labels=warehouse_counts.index,
    autopct='%1.1f%%',
    startangle=140,
    wedgeprops=wedge_props,
    colors=sns.color_palette('Set2', len(warehouse_counts))
)

# Slightly explode the largest slice for emphasis
explode = [0.05 if i == warehouse_counts.argmax() else 0
           for i in range(len(warehouse_counts))]
for wedge, exp in zip(wedges, explode):
    wedge.set_radius(1 + exp)          # manual explode via radius bump

for autotext in autotexts:
    autotext.set_fontsize(10)
    autotext.set_fontweight('bold')

ax.set_title('Order Distribution by Warehouse Block', fontsize=13, pad=16)

plt.tight_layout()
plt.savefig('piechart_warehouse.png', bbox_inches='tight')
plt.show()

print('\n▶ Order counts per Warehouse Block:')
print(warehouse_counts.to_string())

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# BLOCK 8 — Business Insights Summary
# ─────────────────────────────────────────────────────────────────────────────
# All three insights are data-driven; values are computed dynamically so
# the output always reflects the actual loaded dataset.

# --- Insight 1: Customer care calls and late deliveries ---
calls_by_status = df.groupby('Reached.on.Time_Y.N')['Customer_care_calls'].mean()
avg_calls_late   = calls_by_status.get(1, 0)
avg_calls_ontime = calls_by_status.get(0, 0)

# --- Insight 2: Discount and delivery status ---
disc_by_status  = df.groupby('Reached.on.Time_Y.N')['Discount_offered'].median()
med_disc_late   = disc_by_status.get(1, 0)
med_disc_ontime = disc_by_status.get(0, 0)

# --- Insight 3: Dominant warehouse block ---
dominant_block     = warehouse_counts.idxmax()
dominant_pct       = warehouse_counts.max() / warehouse_counts.sum() * 100
late_rate_dominant = (
    df[df['Warehouse_block'] == dominant_block]['Reached.on.Time_Y.N'].mean() * 100
)

insights = f"""
╔══════════════════════════════════════════════════════════════════════════════╗
║              BUSINESS INSIGHTS — E-Commerce Supply Chain EDA               ║
╠══════════════════════════════════════════════════════════════════════════════╣
║                                                                              ║
║  INSIGHT 1 — High Support Calls Signal Delivery Risk                        ║
║  Orders that arrived LATE averaged {avg_calls_late:.2f} customer care calls,          ║
║  versus {avg_calls_ontime:.2f} calls for on-time orders. Logistics teams should         ║
║  flag orders with ≥4 support interactions for proactive intervention.       ║
║                                                                              ║
║  INSIGHT 2 — Large Discounts Correlate with Late Deliveries                 ║
║  Late orders carry a median discount of {med_disc_late:.1f}%, compared to just       ║
║  {med_disc_ontime:.1f}% for on-time orders. Heavily discounted products likely          ║
║  drive demand spikes that overwhelm fulfilment capacity — consider          ║
║  staggering promotional campaigns to flatten order volume peaks.            ║
║                                                                              ║
║  INSIGHT 3 — Warehouse Block {dominant_block} is Handling a Disproportionate Load  ║
║  Block {dominant_block} processes {dominant_pct:.1f}% of all orders with a late-delivery     ║
║  rate of {late_rate_dominant:.1f}%. Redistributing volume to under-utilised blocks      ║
║  or adding picking staff to Block {dominant_block} could reduce overall delays.     ║
║                                                                              ║
╚══════════════════════════════════════════════════════════════════════════════╝
"""
print(insights)